# Section 8 - Operate with OpenQASM- Task 8.1: Export circuits to OpenQASM 3- Task 8.2: Read OpenQASM 3 syntax and semantics- Task 8.3: Interoperate between QuantumCircuit and OpenQASM 3- Task 8.4: Understand Runtime REST API basics

Official references used:- https://openqasm.com/- https://github.com/Qiskit/qiskit/blob/main/qiskit/qasm3/__init__.py- https://github.com/Qiskit/qiskit/blob/main/test/python/qasm3/test_import.py- https://quantum.cloud.ibm.com/apidocs

In [ ]:
from io import StringIOfrom textwrap import dedentimport requestsfrom qiskit import QuantumCircuitimport qiskit.qasm3 as qasm3

## Task 8.1 - Export circuits to OpenQASM 3

The core export API is `qiskit.qasm3.dumps(circuit)` for strings and `qiskit.qasm3.dump(circuit, stream)` for file-like objects.- `include "stdgates.inc";` is typically emitted for standard gates- exporter options such as `includes`, `basis_gates`, and `indent` are passed through keyword arguments

In [ ]:
qc_export = QuantumCircuit(2, 2)qc_export.h(0)qc_export.cx(0, 1)qc_export.measure([0, 1], [0, 1])qasm_text = qasm3.dumps(qc_export)print(qasm_text)

In [ ]:
buffer = StringIO()qasm3.dump(    qc_export,    buffer,    includes=["stdgates.inc"],    basis_gates=["U"],    indent="    ",)print(buffer.getvalue())

## Task 8.2 - Read OpenQASM 3 syntax and semantics

For the exam, distinguish syntax from semantics:- OpenQASM 3 adds typed classical data such as `bit`, `int`, `uint`, `float`, and `angle`- measurement maps quantum data into classical storage- control flow such as `if`, `while`, and custom `gate` declarations changes program meaning, not just formatting

In [ ]:
program = dedent("""OPENQASM 3.0;include "stdgates.inc";input float[64] theta;qubit[2] q;bit[2] mid;gate bell_pair a, b {  h a;  cx a, b;}bell_pair q[0], q[1];mid[0] = measure q[0];if (mid[0] == 1) {  rz(theta) q[1];}mid[1] = measure q[1];""")print(program)

## Task 8.3 - Interoperate between QuantumCircuit and OpenQASM 3

Interop is exam-relevant because export and import support are not symmetric.- `qasm3.dumps` and `qasm3.dump` are part of core Qiskit export- `qasm3.loads` and `qasm3.load` can require the optional `qiskit[qasm3-import]` extra- `loads_experimental` exists for the native experimental importer, but it supports a smaller feature set

In [ ]:
roundtrip_source = qasm3.dumps(qc_export)try:    roundtrip_circuit = qasm3.loads(roundtrip_source)    print("Round-trip qubits:", roundtrip_circuit.num_qubits)    print("Round-trip clbits:", roundtrip_circuit.num_clbits)except Exception as exc:    print("Import requires qasm3 import support and feature compatibility:")    print(type(exc).__name__, exc)

In [ ]:
unsupported_features = [    "annotations or advanced constructs may need annotation handlers",    "unsupported grammar can raise QASM3ImporterError",    "round-trip fidelity depends on the features both sides understand",]for item in unsupported_features:    print("-", item)

## Task 8.4 - Understand Runtime REST API basics

Section 8 also expects basic REST reasoning for Runtime workflows:- `POST` commonly creates a job resource- authentication is typically `Authorization: Bearer <token>`- polling uses job status until a terminal state such as `DONE`, `CANCELLED`, or `ERROR`- cancellation is usually modeled as `POST /v1/jobs/{job_id}/cancel`

In [ ]:
base_url = "https://quantum.cloud.ibm.com/api"headers = {    "Authorization": "Bearer <token>",    "Content-Type": "application/json",}payload = {    "program_id": "sampler",    "backend": "ibm_brisbane",    "params": {"shots": 4096},}submit_request = requests.Request(    "POST",    f"{base_url}/v1/jobs",    headers=headers,    json=payload,).prepare()status_request = requests.Request(    "GET",    f"{base_url}/v1/jobs/<job-id>",    headers=headers,).prepare()cancel_request = requests.Request(    "POST",    f"{base_url}/v1/jobs/<job-id>/cancel",    headers=headers,).prepare()for prepared in (submit_request, status_request, cancel_request):    print(prepared.method, prepared.url)

In [ ]:
terminal_states = {"DONE", "CANCELLED", "ERROR"}sample_statuses = ["QUEUED", "RUNNING", "DONE"]for status in sample_statuses:    print(status, "terminal?", status in terminal_states)

## Quick exam checklist- Use `qiskit.qasm3.dumps` and `dump` for export questions.- Remember that OpenQASM 3 adds typed classical data and control-flow semantics.- Treat `loads` and `load` as import APIs that can depend on optional importer support.- Expect REST questions about `POST`, bearer auth, versioned paths, polling, and cancellation endpoints.